<a href="https://colab.research.google.com/github/vashdev/NeetCode/blob/main/CountriesYouCanSafelyInvestIn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
A telecommunications company wants to expand into new markets. They plan to invest in countries where the average call duration is strictly greater than the global average call duration across all calls.

You are given three tables: person, country, and calls.

person table:
Column Name 	Type
id 	int
name 	varchar
phone_number 	varchar

The id column is the primary key. Each row contains a person's name and phone number. The phone number format is xxx-yyyyyyy where xxx is a 3-digit country code and yyyyyyy is a 7-digit local number.

country table:
Column Name 	Type
name 	varchar
country_code 	varchar

The country_code column is the primary key. Each row maps a country name to its 3-digit code.

calls table:
Column Name 	Type
caller_id 	int
callee_id 	int
duration 	int

This table may contain duplicates. Each row represents a call between two people with the duration in minutes. Note that caller_id is always different from callee_id.

Write a query to find the countries where the average call duration is strictly greater than the global average. A call is associated with a country based on the phone numbers of both the caller and callee.

Return the result in any order.

Example 1:

Input:

person table:
id 	name 	phone_number
3 	David 	051-1234567
12 	Sarah 	051-7654321
1 	Ahmed 	212-1234567
2 	Fatima 	212-6523651
7 	Yosef 	972-1234567
9 	Leah 	972-0011100

country table:
name 	country_code
Peru 	051
Israel 	972
Morocco 	212
Germany 	049
Ethiopia 	251

calls table:
caller_id 	callee_id 	duration
1 	9 	33
2 	9 	4
1 	2 	59
3 	12 	102
3 	12 	330
12 	3 	5
7 	9 	13
7 	1 	3
9 	7 	1
1 	7 	7

Output:
country
Peru
"""


In [7]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Telecom').getOrCreate()
country= spark.createDataFrame(
  [
('Peru', 	'051')
,('Israel', 	'972')
,('Morocco', 	'212')
,('Germany', 	'049')
,('Ethiopia', 	'251')
  ],['name' 	,'country_code']
)

person = spark.createDataFrame(
  [
(3, 'David', '051-1234567'),
(12, 'Sarah', '051-7654321'),
(1, 'Ahmed', '212-1234567'),
(2, 'Fatima', '212-6523651'),
(7, 'Yosef', '972-1234567'),
(9, 'Leah', '972-0011100')
  ],['id', 'name', 'phone_number']
)

calls = spark.createDataFrame(
  [
(1, 9, 33),
(2, 9, 4),
(1, 2, 59),
(3, 12, 102),
(3, 12, 330),
(12, 3, 5),
(7, 9, 13),
(7, 1, 3),
(9, 7, 1),
(1, 7, 7)
  ],['caller_id', 'callee_id', 'duration']
)

In [16]:
from pyspark.sql.functions import *

# Create a mapping from person ID to country name using person and country tables
person_country_map = person.join(country, substr(person.phone_number, lit(1), lit(3)) == country.country_code)\
                           .select(person.id.alias("person_id"), country.name.alias("country_name"))\
                           .distinct()

# Join calls with person_country_map for caller_id
caller_info = calls.join(person_country_map, calls.caller_id == person_country_map.person_id, "inner") \
                   .select(person_country_map.country_name.alias("country"), calls.duration)

# Join calls with person_country_map for callee_id
callee_info = calls.join(person_country_map, calls.callee_id == person_country_map.person_id, "inner") \
                   .select(person_country_map.country_name.alias("country"), calls.duration)

# Union the caller and callee information to get all call durations associated with each country
union_calls = caller_info.unionAll(callee_info)

# Calculate the global average call duration
global_avg_duration = calls.agg(avg("duration")).collect()[0][0]

# Calculate the average call duration for each country
country_avg_duration = union_calls.groupBy("country").agg(avg("duration").alias("avg_duration"))

# Filter countries where their average call duration is strictly greater than the global average
final_result = country_avg_duration.filter(col("avg_duration") > global_avg_duration).select("country")

# Display the result
final_result.show()

+-------+
|country|
+-------+
|   Peru|
+-------+

